# 批量提取所有 absorption 文件夹的结果

本功能扫描 `absorption*`、`mo_absorption*`、`test_*` 开头的文件夹，自动分类到 `1_body` 或 `2_body` 中：
- `absorption_single_*` → `1_body/{元素名}/`
- 其他 → `2_body/{文件夹简写}/`

**mo / test 结构**：无 `job_*` 子目录时，递归查找包含 CONTCAR 的目录（含根目录本身）进行提取。**mo** 源在 GO_final 中输出到带 `mo_` 前缀的文件夹（如 `mo_hollow_H_hollow_H`、`mo_H2O`）。提取 best 时只按元素识别，**mo_** 优先考虑。

提取内容：
- CONTCAR → `{job_name}.vasp`
- OUTCAR → `{job_name}.OUTCAR`（直接复制）
- OSZICAR → `{job_name}.OSZICAR`（复制，用于 E0）
- 所有能量汇总到总表 `all_energies.csv`

**能量来源**：优先使用 OSZICAR 的 **E0**（0K 能量，不含熵），无则用 OUTCAR 的 TOTEN（自由能）。此前 OSZICAR 解析错误地取了 F 而非 E0，已修正。

**比较与恢复**：每次运行会检查目标文件是否存在。若目标已被删除或更名，会重新完成复制；若已存在则覆盖更新（确保与源端一致）。

**mo (modification) 结构**：文件夹名或 job 名以 `mo_` 为前缀表示手动修改的 POSCAR。对于此类结构，会遍历文件夹查找 CONTCAR/OUTCAR 输出，提取结果命名时多加 `mo_` 前缀。最佳结构提取时，若同一组内存在同名结构（一个带 mo_ 一个不带），优先使用带 mo_ 的；输出到 best_structures 时使用基础命名，不再额外标注 mo。

In [4]:
import shutil
import re
from pathlib import Path
import numpy as np
from ase.io import read, write

# ==========================================================
# User controls (edit here)
# ==========================================================
scan_root_dir = Path("/lustre/home/ucaqzh0/thermol/100_water/absorption/absorption/GO/")
output_base_dir = Path("/lustre/home/ucaqzh0/thermol/100_water/absorption/absorption/GO_final")


def read_energy(job: Path):
    """从 OSZICAR 或 OUTCAR 读取能量（job 为包含 OUTCAR/OSZICAR 的目录）
    OSZICAR 格式: '  37 F= -.19505220E+03 E0= -.19504455E+03  d E =...'
    - E0: 0K 能量（不含熵）
    - F: 自由能（与 OUTCAR TOTEN 一致）
    优先读取 E0（常用于 0K 吸附能比较）
    """
    oszicar = job / "OSZICAR"
    if oszicar.exists():
        for line in reversed(oszicar.read_text().splitlines()):
            if "E0=" in line:
                # 正确解析 E0= 后的数值（此前错误地取了 split()[2] 即 F）
                m = re.search(r'E0=\s*([-\d.E+]+)', line)
                if m:
                    try:
                        return float(m.group(1))
                    except (ValueError,):
                        pass

    outcar = job / "OUTCAR"
    if outcar.exists():
        for line in outcar.read_text().splitlines():
            if "free  energy   TOTEN" in line:
                try:
                    return float(line.split()[4])
                except (IndexError, ValueError):
                    pass

    return None


def read_energy_from_outcar(outcar_path: Path):
    """从指定的 OUTCAR 文件路径读取能量（TOTEN，自由能）"""
    if not outcar_path.exists():
        return None
    for line in outcar_path.read_text().splitlines():
        if "free  energy   TOTEN" in line:
            try:
                return float(line.split()[4])
            except (IndexError, ValueError):
                pass
    return None


def read_energy_from_oszicar(oszicar_path: Path):
    """从指定的 OSZICAR 文件路径读取 E0（0K 能量）"""
    if not oszicar_path.exists():
        return None
    for line in reversed(oszicar_path.read_text().splitlines()):
        if "E0=" in line:
            m = re.search(r'E0=\s*([-\d.E+]+)', line)
            if m:
                try:
                    return float(m.group(1))
                except (ValueError,):
                    pass
    return None


def get_folder_short_name(folder_name: str):
    """获取文件夹的简写名称（自动去除 mo_、test_ 前缀）"""
    name = folder_name
    if name.startswith("mo_"):
        name = name[3:]  # 去除 mo_
    if name.startswith("test_2_absorption_single_"):
        return name.replace("test_2_absorption_single_", "")  # test_2_absorption_single_H2O -> H2O
    if name.startswith("test_absorption_"):
        return name.replace("test_absorption_", "")  # test_absorption_H2O -> H2O
    if name.startswith("absorption_single_"):
        return name.replace("absorption_single_", "")
    elif name.startswith("absorption_"):
        return name.replace("absorption_", "")
    else:
        return name


def get_canonical_job_name(short_name: str, third_level, body_type: str, job_name: str) -> str:
    """将 job_name 统一为 job_POSCAR_* 格式（高通量必需，1_body/2_body 命名完全一致）"""
    jn = str(job_name).strip()
    if jn.startswith("job_POSCAR_"):
        return jn
    sn = short_name[3:] if str(short_name).startswith("mo_") else short_name
    tl = "" if (third_level is None or (isinstance(third_level, float) and np.isnan(third_level))) else str(third_level).strip()
    if body_type == "1_body":
        if "absorption_single" in jn or "single_H2O" in jn:
            return f"job_POSCAR_{sn}_single"
        return jn
    if body_type == "2_body":
        mapping = {
            "hollow_H_hollow_H": "job_POSCAR_A_hollow_H__B_hollow_H",
            "hollow_O_hollow_H": "job_POSCAR_A_hollow_O__B_hollow_H",
            "hollow_OH_hollow_H": "job_POSCAR_A_hollow_OH__B_hollow_H",
            "top_OH_bridgeH": "job_POSCAR_A_top_OH__B_bridge_H",
            "top_OH_bridge_H": "job_POSCAR_A_top_OH__B_bridge_H",
            "top_OH_hollow_H": "job_POSCAR_A_top_OH__B_hollow_H",
            "top_OH_topH": "job_POSCAR_A_top_OH__B_top_H",
            "top_OH_top_H": "job_POSCAR_A_top_OH__B_top_H",
        }
        return mapping.get(sn, jn)
    return jn


def find_output_dirs_with_contcar(root: Path):
    """递归遍历文件夹，找到所有包含 CONTCAR 的目录（含根目录本身）"""
    found = []
    if root.is_dir() and (root / "CONTCAR").exists():
        found.append(root)
    for p in root.rglob("*"):
        if p.is_dir() and (p / "CONTCAR").exists():
            found.append(p)
    return found


def find_contcar_dir(job_dir: Path) -> Path:
    """找到包含 CONTCAR 的目录。先检查 job_dir 本身，再遍历子目录。"""
    if (job_dir / "CONTCAR").exists():
        return job_dir
    for sub in job_dir.iterdir():
        if sub.is_dir() and (sub / "CONTCAR").exists():
            return sub
    return job_dir  # 未找到时返回原目录


def extract_third_level_name(job_name: str):
    """从 job_name 中提取第三级文件夹名（位置信息）
    
    例如：
    - job_POSCAR_H_hollow -> hollow
    - job_POSCAR_H_bridge -> bridge
    - job_POSCAR_H_top -> top
    - job_POSCAR_A_hollow_OH__B_hollow_H -> hollow_hollow (简写，只保留位置)
    - job_POSCAR_A_top_OH__B_hollow_H -> top_hollow (简写)
    """
    # 常见的位置关键词
    position_keywords = ['hollow', 'bridge', 'top']
    
    # 移除 mo_ 和 job_POSCAR_ 前缀
    name = job_name
    if name.startswith("mo_"):
        name = name[3:]
    name = name.replace("job_POSCAR_", "")
    
    # 查找位置关键词（保持顺序）
    found_positions = []
    name_lower = name.lower()
    
    for keyword in position_keywords:
        # 检查关键词是否在名称中（作为完整单词）
        # 使用下划线分隔来确保是完整单词
        parts = name_lower.split('_')
        if keyword in parts:
            # 找到所有出现的位置
            for i, part in enumerate(parts):
                if part == keyword:
                    found_positions.append(keyword)
                    break  # 每个关键词只记录一次
    
    if found_positions:
        # 去重但保持顺序
        unique_positions = []
        seen = set()
        for pos in found_positions:
            if pos not in seen:
                unique_positions.append(pos)
                seen.add(pos)
        # 如果有多个位置，用下划线连接（简写）
        return "_".join(unique_positions)
    
    # 如果没有找到位置关键词，返回 None（不创建第三级文件夹）
    return None


def extract_all_absorption_results():
    """批量提取所有 absorption 文件夹的结果"""
    # 创建输出目录结构
    output_base_dir.mkdir(parents=True, exist_ok=True)
    body1_dir = output_base_dir / "1_body"
    body2_dir = output_base_dir / "2_body"
    body1_dir.mkdir(exist_ok=True)
    body2_dir.mkdir(exist_ok=True)
    
    # 扫描 absorption、mo_absorption、test_ 开头的文件夹
    absorption_folders = sorted([
        p for p in scan_root_dir.iterdir() 
        if p.is_dir() and (
            p.name.startswith("absorption") or 
            p.name.startswith("mo_absorption") or 
            p.name.startswith("test_")
        )
    ])
    
    if not absorption_folders:
        print(f"No absorption* folders found under {scan_root_dir}")
        return
    
    print(f"Found {len(absorption_folders)} absorption folder(s).")
    
    all_energy_records = []
    
    for abs_folder in absorption_folders:
        folder_name = abs_folder.name
        print(f"\nProcessing: {folder_name}")
        
        # 判断是 1_body 还是 2_body
        _name = folder_name.replace("mo_", "").replace("test_2_", "").replace("test_", "")
        if "absorption_single_" in _name or _name.startswith("absorption_single_"):
            target_body_dir = body1_dir
            short_name = get_folder_short_name(folder_name)
        else:
            target_body_dir = body2_dir
            short_name = get_folder_short_name(folder_name)
        
        # mo 和 test：遍历查找所有包含 CONTCAR 的目录（可能无 job_*，直接在某级目录）；否则按 job_* 模式扫描
        is_folder_mo = folder_name.startswith("mo_")
        is_folder_test = folder_name.startswith("test_")
        if is_folder_mo or is_folder_test:
            job_dirs = sorted(find_output_dirs_with_contcar(abs_folder))
            tag = "mo" if is_folder_mo else "test"
            print(f"  ({tag}) Traversed, found {len(job_dirs)} output folder(s)")
        else:
            job_dirs = sorted([
                p for p in abs_folder.iterdir() 
                if p.is_dir() and (p.name.startswith("job_") or p.name.startswith("mo_job_"))
            ])
        
        if not job_dirs:
            print(f"  No output folders found in {folder_name}")
            continue
        
        if not is_folder_mo:
            print(f"  Found {len(job_dirs)} job folder(s)")
        
        # 先收集所有 job 的第三级文件夹名
        third_levels = []
        for job in job_dirs:
            job_name = job.name
            third_level = extract_third_level_name(job_name)
            if third_level:
                third_levels.append(third_level)
        
        # 判断是否只有一个唯一的第三级文件夹
        unique_third_levels = set(third_levels)
        use_third_level = len(unique_third_levels) > 1  # 如果有多个不同的第三级，才使用第三级结构
        
        # 处理每个 job 文件夹
        for job in job_dirs:
            job_name = job.name
            is_job_mo = job_name.startswith("mo_")
            
            # mo 结构：输出文件名加 mo_ 前缀（若 job 为文件夹本身则不再重复加）；test 不加前缀
            if is_folder_mo and job == abs_folder:
                output_base_raw = folder_name  # 文件夹本身即为输出，如 mo_absorption_hollow_H_hollow_H
            elif is_folder_mo:
                output_base_raw = f"mo_{job_name}"
            else:
                output_base_raw = job_name
            body_type = "1_body" if target_body_dir == body1_dir else "2_body"
            output_base = get_canonical_job_name(short_name, third_level, body_type, output_base_raw)
            job_path = find_contcar_dir(job) if (is_job_mo and not is_folder_mo and not is_folder_test) else job
            
            # 提取第三级文件夹名（位置信息）
            third_level = extract_third_level_name(job_name)
            
            # 创建目标目录（mo 文件夹在 GO_final 中加 mo_ 前缀）
            target_folder_name = f"mo_{short_name}" if is_folder_mo else short_name
            if third_level and use_third_level:
                target_subdir = target_body_dir / target_folder_name / third_level
            else:
                target_subdir = target_body_dir / target_folder_name
            target_subdir.mkdir(parents=True, exist_ok=True)
            
            # 若已统一为标准命名，删除旧的非标准命名文件
            if output_base != output_base_raw:
                for suf in [".vasp", ".OUTCAR", ".OSZICAR"]:
                    old_f = target_subdir / f"{output_base_raw}{suf}"
                    if old_f.exists():
                        old_f.unlink()
            # 提取 CONTCAR -> .vasp（比较目标是否存在，缺失则重新复制）
            contcar = job_path / "CONTCAR"
            if contcar.exists():
                try:
                    out_vasp = target_subdir / f"{output_base}.vasp"
                    was_missing = not out_vasp.exists()
                    atoms = read(contcar, format="vasp")
                    write(out_vasp, atoms, vasp5=True)
                    tag = " [恢复]" if was_missing else ""
                    print(f"    ✓ {output_base}.vasp -> {target_subdir.relative_to(output_base_dir)}/{tag}")
                except Exception as e:
                    print(f"    ✗ Failed to convert CONTCAR: {output_base} - {e}")
            else:
                print(f"    ⚠ Missing CONTCAR: {output_base}")
            
            # 复制 OUTCAR（比较目标是否存在，缺失则重新复制）
            outcar = job_path / "OUTCAR"
            if outcar.exists():
                try:
                    out_outcar = target_subdir / f"{output_base}.OUTCAR"
                    was_missing = not out_outcar.exists()
                    shutil.copy2(outcar, out_outcar)
                    tag = " [恢复]" if was_missing else ""
                    print(f"    ✓ {output_base}.OUTCAR{tag}")
                except Exception as e:
                    print(f"    ✗ Failed to copy OUTCAR: {output_base} - {e}")
            else:
                print(f"    ⚠ Missing OUTCAR: {output_base}")
            
            # 复制 OSZICAR（用于从 E0 读取能量，0K 吸附能常用）
            oszicar = job_path / "OSZICAR"
            if oszicar.exists():
                try:
                    out_oszicar = target_subdir / f"{output_base}.OSZICAR"
                    shutil.copy2(oszicar, out_oszicar)
                except Exception as e:
                    pass  # 静默，非必须
            
            # 读取能量（从 job_path 源；后面会从 GO_final 重新汇总以更新 CSV）
            energy = read_energy(job_path)
            if energy is not None:
                print(f"    ✓ Energy: {energy:.8f} eV")
            else:
                print(f"    ⚠ No energy found: {output_base}")
    
    # 从 GO_final 实际文件重新汇总能量表（确保 CSV 与输出目录一致，包含恢复/新增的结构）
    all_energy_records = []
    for body_dir, body_type in [(body1_dir, "1_body"), (body2_dir, "2_body")]:
        if not body_dir.exists():
            continue
        for short_dir in sorted(body_dir.iterdir()):
            if not short_dir.is_dir():
                continue
            short_name = short_dir.name
            # 遍历 short_name 下所有 .vasp 文件（可能直接在 short_name 下，或在 third_level 子目录下）
            for vasp_path in short_dir.rglob("*.vasp"):
                # 从路径推断 third_level，再统一为 canonical job_name
                try:
                    rel = vasp_path.relative_to(short_dir)
                    third_level = rel.parent.name if rel.parent != Path(".") else ""
                except Exception:
                    third_level = ""
                job_name = get_canonical_job_name(short_name, third_level, body_type, vasp_path.stem)
                oszicar_path = vasp_path.with_suffix(".OSZICAR")
                outcar_path = vasp_path.with_suffix(".OUTCAR")
                # 优先使用 OSZICAR 的 E0（0K 能量），否则用 OUTCAR 的 TOTEN（自由能）
                energy = read_energy_from_oszicar(oszicar_path)
                if energy is None:
                    energy = read_energy_from_outcar(outcar_path)
                if energy is None:
                    continue
                folder = f"absorption_single_{short_name}" if body_type == "1_body" else f"absorption_{short_name}"
                all_energy_records.append({
                    "folder": folder,
                    "short_name": short_name,
                    "third_level": third_level,
                    "body_type": body_type,
                    "job_name": job_name,
                    "energy_eV": energy
                })
    
    # 保存总能量表（始终覆盖，确保与 GO_final 内容一致）
    energy_csv_path = output_base_dir / "all_energies.csv"
    with energy_csv_path.open("w") as f:
        f.write("folder,short_name,third_level,body_type,job_name,energy_eV\n")
        for record in sorted(all_energy_records, key=lambda r: (r["body_type"], r["short_name"], r["third_level"], r["job_name"])):
            f.write(f"{record['folder']},{record['short_name']},{record['third_level']},"
                   f"{record['body_type']},{record['job_name']},{record['energy_eV']:.8f}\n")
    print(f"\n✓ All energies saved to {energy_csv_path}")
    print(f"  Total records: {len(all_energy_records)}")
    
    print(f"\nDone. Outputs saved to {output_base_dir}")


extract_all_absorption_results()

Found 13 absorption folder(s).

Processing: absorption_hollow_OH_hollow_H
  Found 1 job folder(s)
    ✓ job_POSCAR_A_hollow_OH__B_hollow_H.vasp -> 2_body/hollow_OH_hollow_H/ [恢复]
    ✓ job_POSCAR_A_hollow_OH__B_hollow_H.OUTCAR [恢复]
    ✓ Energy: -206.19298000 eV

Processing: absorption_hollow_O_hollow_H
  Found 1 job folder(s)
    ✓ job_POSCAR_A_hollow_O__B_hollow_H.vasp -> 2_body/hollow_O_hollow_H/ [恢复]
    ✓ job_POSCAR_A_hollow_O__B_hollow_H.OUTCAR [恢复]
    ✓ Energy: -199.31800000 eV

Processing: absorption_single_H
  Found 3 job folder(s)
    ✓ job_POSCAR_H_bridge.vasp -> 1_body/H/bridge/ [恢复]
    ✓ job_POSCAR_H_bridge.OUTCAR [恢复]
    ✓ Energy: -194.95866000 eV
    ✓ job_POSCAR_H_hollow.vasp -> 1_body/H/hollow/ [恢复]
    ✓ job_POSCAR_H_hollow.OUTCAR [恢复]
    ✓ Energy: -195.04455000 eV
    ✓ job_POSCAR_H_top.vasp -> 1_body/H/top/ [恢复]
    ✓ job_POSCAR_H_top.OUTCAR [恢复]
    ✓ Energy: -194.47004000 eV

Processing: absorption_single_H2
  Found 3 job folder(s)
    ✓ job_POSCAR_H2_bridge.v

# 提取最佳结构

根据CSV文件提取能量最低的结构：
- **1_body体系**：每个元素能量最低的结构（mo_H2O 与 H2O 视为同一元素）
- **2_body体系**：识别同种particle组合（如 H、OH_H），提取能量最低的结构
- **mo 优先**：若同一组内存在 mo_ 版本，优先使用
- **best 只识别元素**：输出目录按元素/粒子组合命名（H、H2O、OH_H 等），不含 mo_ 前缀

**比较与恢复**：每次运行会检查目标文件是否存在。若目标已被删除或更名，会重新完成复制；若已存在则覆盖更新。

输出：
- 复制最佳结构的文件到 `best_structures/` 目录
- 生成 `best_structures.csv` 记录最佳结构信息

In [5]:
import pandas as pd
import shutil
import numpy as np
from pathlib import Path
import re

# ==========================================================
# User controls (edit here)
# ==========================================================
csv_path = Path("/lustre/home/ucaqzh0/thermol/100_water/absorption/absorption/GO_final/all_energies.csv")
source_base_dir = Path("/lustre/home/ucaqzh0/thermol/100_water/absorption/absorption/GO_final")
output_best_dir = Path("/lustre/home/ucaqzh0/thermol/100_water/absorption/absorption/best_structures")


def element_from_short_name(short_name: str) -> str:
    """从 short_name 提取元素（best_structures 只按元素识别，mo_ 前缀不参与分组）"""
    return short_name[3:] if short_name.startswith("mo_") else short_name


def get_canonical_job_name(short_name: str, third_level, body_type: str, job_name: str) -> str:
    """将 job_name 统一为 job_POSCAR_* 格式（高通量必需）"""
    jn = str(job_name).strip()
    if jn.startswith("job_POSCAR_"):
        return jn
    sn = short_name[3:] if str(short_name).startswith("mo_") else short_name
    tl = "" if (third_level is None or (isinstance(third_level, float) and np.isnan(third_level))) else str(third_level).strip()
    if body_type == "1_body":
        if "absorption_single" in jn or "single_H2O" in jn:
            return f"job_POSCAR_{sn}_single"
        return jn
    if body_type == "2_body":
        mapping = {
            "hollow_H_hollow_H": "job_POSCAR_A_hollow_H__B_hollow_H",
            "hollow_O_hollow_H": "job_POSCAR_A_hollow_O__B_hollow_H",
            "hollow_OH_hollow_H": "job_POSCAR_A_hollow_OH__B_hollow_H",
            "top_OH_bridgeH": "job_POSCAR_A_top_OH__B_bridge_H",
            "top_OH_bridge_H": "job_POSCAR_A_top_OH__B_bridge_H",
            "top_OH_hollow_H": "job_POSCAR_A_top_OH__B_hollow_H",
            "top_OH_topH": "job_POSCAR_A_top_OH__B_top_H",
            "top_OH_top_H": "job_POSCAR_A_top_OH__B_top_H",
        }
        return mapping.get(sn, jn)
    return jn


def extract_particle_combination(short_name: str):
    """从2_body的short_name中提取particle组合
    
    例如：
    - hollow_H_hollow_H -> H_H
    - mo_hollow_H_hollow_H -> H_H（自动去除 mo_）
    """
    short_name = short_name[3:] if short_name.startswith("mo_") else short_name
    # 特殊处理：chemical 对应 H2O
    if short_name.lower() == 'chemical':
        return 'H2O'
    
    # 粒子类型列表（按长度从长到短排序，先匹配长的避免误匹配）
    particle_types = ['H2O', 'OH', 'H2', 'O', 'H']
    
    # 在名称中搜索所有粒子类型（不区分大小写）
    name_upper = short_name.upper()
    found_particles = []
    
    # 对于每个粒子类型，检查是否在名称中出现
    for ptype in particle_types:
        ptype_upper = ptype.upper()
        # 检查粒子类型是否在名称中出现
        # 1. 作为独立单词（用下划线分隔）：_OH_, _OH, OH_
        # 2. 在位置关键词后面：bridgeH, topH, hollowH
        # 3. 在位置关键词前面：H_bridge, H_top
        if ptype_upper in name_upper:
            # 检查是否是独立单词（用下划线分隔）
            if f'_{ptype_upper}_' in name_upper or \
               name_upper.startswith(f'{ptype_upper}_') or \
               name_upper.endswith(f'_{ptype_upper}'):
                found_particles.append(ptype)
            # 检查是否在位置关键词后面（如 bridgeH, topH, hollowH）
            elif 'BRIDGE' + ptype_upper in name_upper or \
                 'TOP' + ptype_upper in name_upper or \
                 'HOLLOW' + ptype_upper in name_upper:
                found_particles.append(ptype)
            # 检查是否在位置关键词前面（如 H_bridge, H_top）
            elif ptype_upper + '_BRIDGE' in name_upper or \
                 ptype_upper + '_TOP' in name_upper or \
                 ptype_upper + '_HOLLOW' in name_upper:
                found_particles.append(ptype)
    
    # 去重并排序
    if found_particles:
        unique_particles = []
        seen = set()
        for p in found_particles:
            if p not in seen:
                unique_particles.append(p)
                seen.add(p)
        # 排序：OH 在 H 之前，其他按字母顺序
        # 使用自定义排序键：OH -> 'A', H -> 'B', 其他保持原值
        sorted_particles = sorted(unique_particles, key=lambda x: 'A' if x == 'OH' else ('B' if x == 'H' else x))
        return '_'.join(sorted_particles)
    
    return short_name  # 如果无法提取，返回原始名称


def job_base_name(job_name: str) -> str:
    """去除 job_name 的 mo_ 前缀，得到基础命名"""
    return job_name[3:] if job_name.startswith("mo_") else job_name


def filter_prefer_mo(group: pd.DataFrame) -> pd.DataFrame:
    """同组内若存在 mo_ 版本，则移除对应的非 mo 版本"""
    job_names = group['job_name'].astype(str)
    bases_with_mo = {job_base_name(jn) for jn in job_names[job_names.str.startswith('mo_')]}
    if not bases_with_mo:
        return group
    non_mo_bases = job_names.apply(job_base_name)
    remove = (~job_names.str.startswith('mo_')) & (non_mo_bases.isin(bases_with_mo))
    return group[~remove]


def extract_best_structures():
    """提取最佳结构"""
    # 读取CSV文件
    df = pd.read_csv(csv_path)
    
    # 移除能量为空的行
    df = df[df['energy_eV'].notna()]
    
    # 创建输出目录
    output_best_dir.mkdir(parents=True, exist_ok=True)
    
    best_records = []
    
    # 处理1_body体系
    body1_df = df[df['body_type'] == '1_body'].copy()
    if not body1_df.empty:
        print("Processing 1_body systems...")
        # 按元素分组（best 只识别元素，mo_H2O 与 H2O 视为同一元素），找每个元素能量最低的结构（mo 优先）
        body1_df['_element'] = body1_df['short_name'].apply(element_from_short_name)
        for element, group in body1_df.groupby('_element'):
            group = filter_prefer_mo(group)
            best_idx = group['energy_eV'].idxmin()
            best_row = group.loc[best_idx]
            
            source_job_name = best_row['job_name']  # 用于查找源文件（可能含 mo_）
            short_name = best_row['short_name']  # GO_final 中可能为 mo_H2O
            third_level = best_row['third_level']
            best_job_name = get_canonical_job_name(short_name, third_level, '1_body', source_job_name)  # 统一为 job_POSCAR_*
            print(f"  {element}: {source_job_name} (E={best_row['energy_eV']:.8f} eV)")
            
            # 构建源文件路径（short_name 可能带 mo_ 前缀，如 1_body/mo_H2O/）
            job_name = source_job_name
            source_vasp = None
            source_outcar = None
            
            # 尝试路径1：有 third_level（安全处理 NaN/空值）
            third_level = best_row['third_level']
            has_third = pd.notna(third_level) and str(third_level).strip()
            if has_third:
                source_dir = source_base_dir / "1_body" / short_name / str(third_level)
                vasp_path = source_dir / f"{job_name}.vasp"
                outcar_path = source_dir / f"{job_name}.OUTCAR"
                if vasp_path.exists():
                    source_vasp = vasp_path
                    source_outcar = outcar_path if outcar_path.exists() else None
            
            # 尝试路径2：没有 third_level
            if source_vasp is None:
                source_dir = source_base_dir / "1_body" / short_name
                vasp_path = source_dir / f"{job_name}.vasp"
                outcar_path = source_dir / f"{job_name}.OUTCAR"
                if vasp_path.exists():
                    source_vasp = vasp_path
                    source_outcar = outcar_path if outcar_path.exists() else None
            
            # 只有在文件存在时才创建文件夹和复制文件（best 只识别元素，不含 mo_）
            if source_vasp and source_vasp.exists():
                target_dir = output_best_dir / "1_body" / element
                target_dir.mkdir(parents=True, exist_ok=True)
                # 清理该目录下旧的 .vasp/.OUTCAR/.OSZICAR（避免最佳变更后残留旧文件）
                for old in list(target_dir.glob("*.vasp")) + list(target_dir.glob("*.OUTCAR")) + list(target_dir.glob("*.OSZICAR")):
                    if old.name not in (f"{best_job_name}.vasp", f"{best_job_name}.OUTCAR", f"{best_job_name}.OSZICAR"):
                        old.unlink()
                
                # 复制文件（best 使用基础命名，不含 mo_）；比较目标是否存在，缺失则重新复制
                target_vasp = target_dir / f"{best_job_name}.vasp"
                was_missing = not target_vasp.exists()
                shutil.copy2(source_vasp, target_vasp)
                tag = " [恢复]" if was_missing else ""
                print(f"    ✓ Copied {best_job_name}.vasp{tag}")
                
                if source_outcar and source_outcar.exists():
                    target_outcar = target_dir / f"{best_job_name}.OUTCAR"
                    was_missing_o = not target_outcar.exists()
                    shutil.copy2(source_outcar, target_outcar)
                    if was_missing_o:
                        print(f"    ✓ Copied {best_job_name}.OUTCAR [恢复]")
                source_oszicar = source_vasp.with_suffix(".OSZICAR")
                if source_oszicar.exists():
                    shutil.copy2(source_oszicar, target_dir / f"{best_job_name}.OSZICAR")
                
                best_records.append({
                    'body_type': '1_body',
                    'particle_combination': element,
                    'short_name': short_name,
                    'third_level': best_row['third_level'],
                    'job_name': best_job_name,
                    'energy_eV': best_row['energy_eV'],
                    'folder': best_row['folder']
                })
            else:
                print(f"    ⚠ Source file not found: {source_job_name}.vasp")
    
    # 处理2_body体系
    body2_df = df[df['body_type'] == '2_body'].copy()
    if not body2_df.empty:
        print("\nProcessing 2_body systems...")
        # 提取particle组合
        body2_df['particle_combination'] = body2_df['short_name'].apply(extract_particle_combination)
        
        # 按particle组合分组，找每个组合能量最低的结构（mo 优先）
        for combination, group in body2_df.groupby('particle_combination'):
            group = filter_prefer_mo(group)
            best_idx = group['energy_eV'].idxmin()
            best_row = group.loc[best_idx]
            
            source_job_name = best_row['job_name']  # 用于查找源文件（可能含 mo_）
            short_name = best_row['short_name']
            third_level = best_row['third_level']
            best_job_name = get_canonical_job_name(short_name, third_level, '2_body', source_job_name)  # 统一为 job_POSCAR_*
            print(f"  {combination}: {source_job_name} (E={best_row['energy_eV']:.8f} eV)")
            
            # 查找源文件（使用 short_name 精确定位，避免同名 job 在不同文件夹时取错）
            job_name = source_job_name
            source_vasp = None
            source_outcar = None
            has_third = pd.notna(third_level) and str(third_level).strip()
            
            source_folder = source_base_dir / "2_body" / short_name
            if source_folder.exists():
                # 尝试路径1：有 third_level
                if has_third:
                    vasp_path = source_folder / str(third_level) / f"{job_name}.vasp"
                    outcar_path = source_folder / str(third_level) / f"{job_name}.OUTCAR"
                    if vasp_path.exists():
                        source_vasp = vasp_path
                        source_outcar = outcar_path if outcar_path.exists() else None
                # 尝试路径2：没有 third_level
                if source_vasp is None:
                    vasp_path = source_folder / f"{job_name}.vasp"
                    outcar_path = source_folder / f"{job_name}.OUTCAR"
                    if vasp_path.exists():
                        source_vasp = vasp_path
                        source_outcar = outcar_path if outcar_path.exists() else None
            
            # 只有在文件存在时才创建文件夹和复制文件（best 输出用基础命名）
            if source_vasp and source_vasp.exists():
                # 构建目标路径（使用particle组合作为文件夹名）
                target_dir = output_best_dir / "2_body" / combination
                target_dir.mkdir(parents=True, exist_ok=True)
                # 清理该目录下旧的 .vasp/.OUTCAR/.OSZICAR（避免最佳变更后残留旧文件）
                for old in list(target_dir.glob("*.vasp")) + list(target_dir.glob("*.OUTCAR")) + list(target_dir.glob("*.OSZICAR")):
                    if old.name not in (f"{best_job_name}.vasp", f"{best_job_name}.OUTCAR", f"{best_job_name}.OSZICAR"):
                        old.unlink()
                
                # 复制文件（best 使用基础命名，不含 mo_）；比较目标是否存在，缺失则重新复制
                target_vasp = target_dir / f"{best_job_name}.vasp"
                was_missing = not target_vasp.exists()
                shutil.copy2(source_vasp, target_vasp)
                tag = " [恢复]" if was_missing else ""
                print(f"    ✓ Copied {best_job_name}.vasp{tag}")
                
                if source_outcar and source_outcar.exists():
                    target_outcar = target_dir / f"{best_job_name}.OUTCAR"
                    was_missing_o = not target_outcar.exists()
                    shutil.copy2(source_outcar, target_outcar)
                    if was_missing_o:
                        print(f"    ✓ Copied {best_job_name}.OUTCAR [恢复]")
                source_oszicar = source_vasp.with_suffix(".OSZICAR")
                if source_oszicar.exists():
                    shutil.copy2(source_oszicar, target_dir / f"{best_job_name}.OSZICAR")
                
                best_records.append({
                    'body_type': '2_body',
                    'particle_combination': combination,
                    'short_name': best_row['short_name'],
                    'third_level': best_row['third_level'],
                    'job_name': best_job_name,
                    'energy_eV': best_row['energy_eV'],
                    'folder': best_row['folder']
                })
            else:
                print(f"    ⚠ Source file not found: {source_job_name}.vasp")
    
    # 保存最佳结构记录
    if best_records:
        best_df = pd.DataFrame(best_records)
        best_csv_path = output_best_dir / "best_structures.csv"
        best_df.to_csv(best_csv_path, index=False)
        print(f"\n✓ Best structures saved to {output_best_dir}")
        print(f"✓ Best structures CSV saved to {best_csv_path}")
        print(f"  Total best structures: {len(best_records)}")
    else:
        print("\n⚠ No best structures found")


extract_best_structures()

Processing 1_body systems...
  H: job_POSCAR_H_hollow (E=-195.04455000 eV)
    ✓ Copied job_POSCAR_H_hollow.vasp [恢复]
    ✓ Copied job_POSCAR_H_hollow.OUTCAR [恢复]
  H2: job_POSCAR_H2_top (E=-198.26102000 eV)
    ✓ Copied job_POSCAR_H2_top.vasp [恢复]
    ✓ Copied job_POSCAR_H2_top.OUTCAR [恢复]
  H2O: job_POSCAR_H2O_single (E=-206.08342000 eV)
    ✓ Copied job_POSCAR_H2O_single.vasp [恢复]
    ✓ Copied job_POSCAR_H2O_single.OUTCAR [恢复]
  O: job_POSCAR_O_hollow (E=-198.57034000 eV)
    ✓ Copied job_POSCAR_O_hollow.vasp [恢复]
    ✓ Copied job_POSCAR_O_hollow.OUTCAR [恢复]
  OH: job_POSCAR_OH_hollow (E=-202.61904000 eV)
    ✓ Copied job_POSCAR_OH_hollow.vasp [恢复]
    ✓ Copied job_POSCAR_OH_hollow.OUTCAR [恢复]

Processing 2_body systems...
  H: job_POSCAR_A_hollow_H__B_hollow_H (E=-198.62755000 eV)
    ✓ Copied job_POSCAR_A_hollow_H__B_hollow_H.vasp [恢复]
    ✓ Copied job_POSCAR_A_hollow_H__B_hollow_H.OUTCAR [恢复]
  H2O: job_POSCAR_H2O_bridge_custom (E=-206.07292000 eV)
    ✓ Copied job_POSCAR_H2O_bri